# Подготовка датасета обращений и SLA-анализ

Ноутбук читает один исходный CSV, сохраняет исходные категории, рассчитывает исправленные процессные признаки, формирует `final.csv` и выполняет проверки качества.

In [1]:
from pathlib import Path
import os
import re
from typing import Optional, Dict, Any

import numpy as np
import pandas as pd

INPUT_CSV_PATH = Path(os.getenv("INPUT_CSV_PATH", "modules/routing/reports/routing/shared_split/appeals_prepared.csv"))
OUTPUT_CSV_PATH = Path("final.csv")
DATETIME_SMALL_NEGATIVE_TOLERANCE_MIN = 5
OTHER_LABEL = "Прочее"

print(f"Источник: {INPUT_CSV_PATH}")
print(f"Результат: {OUTPUT_CSV_PATH}")

Источник: modules/routing/reports/routing/shared_split/appeals_prepared.csv
Результат: final.csv


## Преобразование исходных данных

Исходная категория хранится в `category_original`. Укрупнение top-15 выполняется отдельно в `category_grouped`.

In [2]:
def parse_duration_to_hours(val: Any) -> float:
    """Преобразует H:MM[:SS] в часы, сохраняя настоящий пропуск как NaN."""
    if pd.isna(val):
        return np.nan
    value = str(val).strip()
    if not value:
        return np.nan
    negative = value.startswith("-")
    if negative:
        value = value[1:]
    parts = value.split(":")
    try:
        nums = [float(part) for part in parts]
    except (TypeError, ValueError):
        return np.nan
    if len(nums) == 3:
        hours = nums[0] + nums[1] / 60 + nums[2] / 3600
    elif len(nums) == 2:
        hours = nums[0] + nums[1] / 60
    else:
        return np.nan
    return -hours if negative else hours


def _clean_text(series: pd.Series, missing: Optional[str] = None) -> pd.Series:
    result = series.astype("string").str.strip().str.replace(r"\s+", " ", regex=True)
    return result.fillna(missing) if missing is not None else result


def process_tickets_dataset(csv_path: Path) -> pd.DataFrame:
    raw = pd.read_csv(csv_path, encoding="utf-8-sig")
    clean = pd.DataFrame(index=raw.index)

    clean["ticket_id"] = _clean_text(raw["Номер запроса"])
    date_map = {
        "reg_dt": "Дата регистрации",
        "plan_dt": "Плановое время выполнения",
        "deadline_dt": "Крайний срок обработки",
        "fact_dt": "Фактическое время выполнения",
        "status_entry_dt": "Время входа в статус",
        "last_modified_dt": "Дата последнего изменения",
    }
    for target, source in date_map.items():
        clean[target] = pd.to_datetime(raw[source], errors="coerce", format="mixed", dayfirst=True)

    text_map = {
        "status": "Статус", "user_type": "Пользователь", "service": "Услуга",
        "request_type": "Тип запроса", "criticality": "Критичность",
        "urgency": "Срочность", "priority": "Приоритет",
        "service_class": "Класс обслуживания", "timezone": "Часовой пояс запроса",
        "resolution": "Результат работ", "resolved_line": "Кем решен (группа)",
    }
    for target, source in text_map.items():
        clean[target] = _clean_text(raw[source])
    clean["service_component"] = _clean_text(raw["Компонент услуги 1 уровня"], "Не указан")
    clean["text_clean"] = _clean_text(raw["Описание 2"])
    clean["category_original"] = _clean_text(raw["Вид запроса"])

    meaningful_counts = clean.loc[clean["category_original"] != OTHER_LABEL, "category_original"].value_counts()
    top_15_categories = meaningful_counts.head(15).index
    clean["category_grouped"] = clean["category_original"].where(
        clean["category_original"].isin(top_15_categories), OTHER_LABEL
    )

    clean["norm_sla_h"] = raw["Норматив ное время обработки (SLA)"].apply(parse_duration_to_hours)
    source_fact_sla = raw["Фактическая длительность выполнения запроса (SLA)"].apply(parse_duration_to_hours)

    line_map = {
        "line_1_react_h": "Суммарное время реакции 1 линии", "line_1_work_h": "Суммарное время работы 1 линии",
        "line_2_react_h": "Суммарное время реакции 2 линии", "line_2_work_h": "Суммарное время работы 2 линии",
        "line_3_react_h": "Суммарное время реакции 3 линии", "line_3_work_h": "Суммарное время работы 3 линии",
        "line_4_react_h": "Суммарное время реакции 4 линии", "line_4_work_h": "Суммарное время работы 4 линии",
    }
    for target, source in line_map.items():
        clean[target] = raw[source].apply(parse_duration_to_hours).fillna(0.0)
    clean["clarification_h"] = raw["Суммарное время уточнений"].apply(parse_duration_to_hours).fillna(0.0)

    react_cols = [f"line_{i}_react_h" for i in range(1, 5)]
    work_cols = [f"line_{i}_work_h" for i in range(1, 5)]
    line_sum = clean[react_cols + work_cols].sum(axis=1)
    can_reconstruct = source_fact_sla.isna() & line_sum.gt(0)
    clean["fact_sla_h"] = source_fact_sla.where(source_fact_sla.notna(), line_sum.where(can_reconstruct))
    clean["fact_sla_source"] = np.select(
        [source_fact_sla.notna(), can_reconstruct],
        ["source", "reconstructed_from_lines"],
        default="missing",
    )

    clean["clarifications_cnt"] = pd.to_numeric(raw["Количество уточнений"], errors="coerce").astype("Int64")
    clean["total_work_h"] = clean[work_cols].sum(axis=1)
    clean["total_react_h"] = clean[react_cols].sum(axis=1)
    old_lines_count = clean[work_cols].gt(0).sum(axis=1)
    active_lines = pd.DataFrame({
        f"line_{i}": clean[f"line_{i}_react_h"].gt(0) | clean[f"line_{i}_work_h"].gt(0)
        for i in range(1, 5)
    })
    clean["lines_count"] = active_lines.sum(axis=1).astype(int)
    clean["is_multiline"] = clean["lines_count"].gt(1).astype(int)
    clean["lines_count_changed"] = clean["lines_count"].ne(old_lines_count).astype(int)
    clean["is_overdue"] = _clean_text(raw["Просрочен?*"]).eq("Просрочен").astype(int)
    clean["is_high_clarifications"] = clean["clarifications_cnt"].ge(2).fillna(False).astype(int)

    lifecycle_h = (clean["fact_dt"] - clean["reg_dt"]).dt.total_seconds() / 3600
    negative = lifecycle_h.lt(0)
    small_negative = negative & lifecycle_h.ge(-DATETIME_SMALL_NEGATIVE_TOLERANCE_MIN / 60)
    clean["has_datetime_anomaly"] = negative.astype(int)
    clean["total_lifecycle_h"] = lifecycle_h.mask(small_negative, 0.0).mask(negative & ~small_negative, np.nan)

    output_order = ['ticket_id', 'reg_dt', 'plan_dt', 'deadline_dt', 'fact_dt', 'status_entry_dt', 'last_modified_dt', 'status', 'user_type', 'service', 'service_component', 'category_original', 'category_grouped', 'request_type', 'criticality', 'urgency', 'priority', 'service_class', 'timezone', 'text_clean', 'norm_sla_h', 'fact_sla_h', 'fact_sla_source', 'line_1_react_h', 'line_1_work_h', 'line_2_react_h', 'line_2_work_h', 'line_3_react_h', 'line_3_work_h', 'line_4_react_h', 'line_4_work_h', 'clarification_h', 'resolution', 'resolved_line', 'clarifications_cnt', 'total_work_h', 'total_react_h', 'lines_count', 'is_multiline', 'lines_count_changed', 'is_overdue', 'is_high_clarifications', 'total_lifecycle_h', 'has_datetime_anomaly']
    return clean[output_order]


clean_df = process_tickets_dataset(INPUT_CSV_PATH)
assert clean_df["category_original"].nunique() == 43, "Ожидалось 43 исходные категории"
assert (clean_df["total_lifecycle_h"].dropna() >= 0).all(), "Отрицательные длительности не устранены"
assert set(clean_df["fact_sla_source"].unique()) <= {"source", "reconstructed_from_lines", "missing"}

clean_df.to_csv(OUTPUT_CSV_PATH, index=False, encoding="utf-8-sig", date_format="%Y-%m-%d %H:%M:%S.%f")
print(f"Сохранено строк: {len(clean_df):,}")
print(f"Исходных категорий: {clean_df['category_original'].nunique()}")
print(f"Укрупнённых категорий: {clean_df['category_grouped'].nunique()}")
print(f"Изменился lines_count: {clean_df['lines_count_changed'].sum():,}")

Сохранено строк: 1,931
Исходных категорий: 43
Укрупнённых категорий: 16
Изменился lines_count: 98


In [3]:
REGISTRATION_COLS = [
    "ticket_id", "reg_dt", "service", "service_component", "category_original", "category_grouped",
    "request_type", "criticality", "urgency", "priority", "service_class", "norm_sla_h",
    "plan_dt", "deadline_dt", "text_clean"
]
WORKFLOW_COLS = [
    "ticket_id", "fact_dt", "status_entry_dt", "last_modified_dt", "fact_sla_h", "fact_sla_source",
    "total_lifecycle_h", "has_datetime_anomaly", "is_overdue", "resolved_line", "clarifications_cnt",
    "clarification_h", "is_high_clarifications", "line_1_react_h", "line_1_work_h",
    "line_2_react_h", "line_2_work_h", "line_3_react_h", "line_3_work_h",
    "line_4_react_h", "line_4_work_h", "total_work_h", "total_react_h", "lines_count",
    "is_multiline", "lines_count_changed"
]


def split_features(df: pd.DataFrame):
    """Разделяет регистрационные признаки и итоговые метрики выполнения."""
    return df[REGISTRATION_COLS].copy(), df[WORKFLOW_COLS].copy()


def filter_tickets(
    df: pd.DataFrame,
    date_from: Optional[str] = None,
    date_to: Optional[str] = None,
    service: Optional[str] = None,
    category: Optional[str] = None,
    priority: Optional[str] = None,
    line: Optional[str] = None,
) -> pd.DataFrame:
    """Фильтрует обращения; YYYY-MM-DD в date_to включает весь указанный день."""
    filtered = df.copy()
    if date_from:
        filtered = filtered[filtered["reg_dt"] >= pd.to_datetime(date_from)]
    if date_to:
        upper = pd.to_datetime(date_to)
        if re.fullmatch(r"\d{4}-\d{2}-\d{2}", str(date_to)):
            filtered = filtered[filtered["reg_dt"] < upper + pd.Timedelta(days=1)]
        else:
            filtered = filtered[filtered["reg_dt"] <= upper]
    if service and service != "Все":
        filtered = filtered[filtered["service"] == service]
    if category and category != "Все":
        category_col = "category_original" if category in set(df["category_original"].dropna()) else "category_grouped"
        filtered = filtered[filtered[category_col] == category]
    if priority and priority != "Все":
        filtered = filtered[filtered["priority"] == priority]
    if line and line != "Все":
        filtered = filtered[filtered["resolved_line"] == line]
    return filtered


def get_summary_kpis(df: pd.DataFrame) -> Dict[str, Any]:
    total = len(df)
    if total == 0:
        return {}
    overdue = int(df["is_overdue"].sum())
    return {
        "Всего обращений": total,
        "Просрочено обращений": overdue,
        "Доля просрочки (%)": round(overdue / total * 100, 2),
        "Средняя длительность SLA (ч)": round(df["fact_sla_h"].mean(), 2),
        "Медианная длительность SLA (ч)": round(df["fact_sla_h"].median(), 2),
        "Доля обращений через несколько линий (%)": round(df["is_multiline"].mean() * 100, 2),
        "Среднее число уточнений": round(df["clarifications_cnt"].mean(), 2),
    }


# Граничный тест: в выборку входит обращение с временем в конце date_to.
_filter_probe = pd.DataFrame({"reg_dt": pd.to_datetime(["2025-01-01 00:00:00", "2025-01-01 23:59:59.999999", "2025-01-02"], format="mixed"),
                              "service": ["x"] * 3, "category_original": ["x"] * 3,
                              "category_grouped": ["x"] * 3, "priority": ["x"] * 3, "resolved_line": ["x"] * 3})
assert len(filter_tickets(_filter_probe, date_to="2025-01-01")) == 2
print("Граничный тест date_to: пройден")

Граничный тест date_to: пройден


## SLA-анализ

Риск-скор вычисляется одинаково для каждого разреза: `1.5 × доля просрочки, % + 0.05 × доля обращений через несколько линий, % + 0.1 × средний SLA, ч`. Средние и медианы игнорируют отсутствующий SLA.

In [4]:
def get_metrics_by_dimension(df: pd.DataFrame, dimension: str) -> pd.DataFrame:
    """Возвращает единый набор SLA-метрик для одного измерения."""
    stats = df.groupby(dimension, dropna=False).agg(
        tickets_count=("ticket_id", "count"),
        overdue_count=("is_overdue", "sum"),
        overdue_share_pct=("is_overdue", lambda x: x.mean() * 100),
        mean_sla_h=("fact_sla_h", "mean"),
        median_sla_h=("fact_sla_h", "median"),
        multiline_share_pct=("is_multiline", lambda x: x.mean() * 100),
        mean_clarifications=("clarifications_cnt", "mean"),
    ).reset_index()
    stats["risk_score"] = (
        stats["overdue_share_pct"] * 1.5
        + stats["multiline_share_pct"] * 0.05
        + stats["mean_sla_h"].fillna(0) * 0.1
    )
    numeric = ["overdue_share_pct", "mean_sla_h", "median_sla_h", "multiline_share_pct", "mean_clarifications", "risk_score"]
    stats[numeric] = stats[numeric].round(2)
    return stats.sort_values(["risk_score", "tickets_count"], ascending=[False, False]).reset_index(drop=True)


def get_sla_risk_matrix(df: pd.DataFrame, dimension: str = "category_original", min_tickets: int = 20) -> pd.DataFrame:
    """Совместимая функция риск-матрицы для любого поддерживаемого разреза."""
    return get_metrics_by_dimension(df, dimension).query("tickets_count >= @min_tickets")


SLA_DIMENSIONS = {
    "Исходные категории": "category_original",
    "Категории top-15": "category_grouped",
    "Линия решения": "resolved_line",
    "Услуга": "service",
    "Приоритет": "priority",
}
sla_analysis = {label: get_metrics_by_dimension(clean_df, column) for label, column in SLA_DIMENSIONS.items()}

for label, table in sla_analysis.items():
    print(f"\n{label} ({len(table)} групп)")
    display(table)


Исходные категории (43 групп)
                                            category_original  tickets_count  overdue_count  overdue_share_pct  mean_sla_h  median_sla_h  multiline_share_pct  mean_clarifications  risk_score
0                               Интеграционное взаимодействие              3              1              33.33      124.31         13.66               100.00                  1.0       67.43
1                                           Проблема из почты              9              2              22.22       17.15          5.54                77.78                 0.33       38.94
2                            Проблема с загрузкой данных в 1С             18              1               5.56       17.99          2.15                83.33                 0.72       14.30
3                          Ошибки загрузки страницы/зависания             66              4               6.06        6.43          1.77                87.88                  1.8       14.13
4             

## Словарь данных

Полная таблица полей приведена ниже и сохранена отдельным файлом `data_dictionary.md`.

# Словарь данных итогового датасета

`final.csv` формируется из исходного CSV параметром `INPUT_CSV_PATH`. Пропуски SLA не заменяются нулём.

| Поле | Источник или формула | Тип | Смысл |
|---|---|---|---|
| `ticket_id` | Номер запроса | `string` | Уникальный идентификатор обращения. |
| `reg_dt` | Дата регистрации | `datetime64[ns]` | Дата и время регистрации обращения. |
| `plan_dt` | Плановое время выполнения | `datetime64[ns]` | Плановое время выполнения из источника. |
| `deadline_dt` | Крайний срок обработки | `datetime64[ns]` | Крайний срок обработки из источника. |
| `fact_dt` | Фактическое время выполнения | `datetime64[ns]` | Фактическое время завершения; исходное значение сохраняется даже при аномалии. |
| `status_entry_dt` | Время входа в статус | `datetime64[ns]` | Время последнего входа в текущий статус. |
| `last_modified_dt` | Дата последнего изменения | `datetime64[ns]` | Дата последнего изменения записи. |
| `status` | Статус | `string` | Текущий статус обращения. |
| `user_type` | Пользователь | `string` | Тип пользователя, создавшего обращение. |
| `service` | Услуга | `string` | Услуга, к которой относится обращение. |
| `service_component` | Компонент услуги 1 уровня | `string` | Компонент услуги первого уровня; пропуски помечены «Не указан». |
| `category_original` | Вид запроса | `string` | Исходная категория без укрупнения; сохраняет все 43 категории. |
| `category_grouped` | Формула top-15 | `string` | 15 наиболее частых категорий без «Прочее»; остальные объединены в «Прочее». |
| `request_type` | Тип запроса | `string` | Тип обращения. |
| `criticality` | Критичность | `string` | Критичность обращения. |
| `urgency` | Срочность | `string` | Срочность обращения. |
| `priority` | Приоритет | `string` | Приоритет обработки. |
| `service_class` | Класс обслуживания | `string` | Класс обслуживания и календарь SLA. |
| `timezone` | Часовой пояс запроса | `string` | Часовой пояс обращения. |
| `text_clean` | Описание 2 | `string` | Текст обращения с обрезанными краевыми пробелами. |
| `norm_sla_h` | Нормативное время обработки (SLA) | `float64` | Норматив SLA в часах; пропуск не заменяется нулём. |
| `fact_sla_h` | Источник или сумма времен линий | `float64` | Фактический SLA в часах; отсутствует, если его нельзя восстановить. |
| `fact_sla_source` | Формула происхождения SLA | `string` | source, reconstructed_from_lines или missing. |
| `line_1_react_h` | Суммарное время реакции 1 линии | `float64` | Время реакции первой линии, часы. |
| `line_1_work_h` | Суммарное время работы 1 линии | `float64` | Время работы первой линии, часы. |
| `line_2_react_h` | Суммарное время реакции 2 линии | `float64` | Время реакции второй линии, часы. |
| `line_2_work_h` | Суммарное время работы 2 линии | `float64` | Время работы второй линии, часы. |
| `line_3_react_h` | Суммарное время реакции 3 линии | `float64` | Время реакции третьей линии, часы. |
| `line_3_work_h` | Суммарное время работы 3 линии | `float64` | Время работы третьей линии, часы. |
| `line_4_react_h` | Суммарное время реакции 4 линии | `float64` | Время реакции четвёртой линии, часы. |
| `line_4_work_h` | Суммарное время работы 4 линии | `float64` | Время работы четвёртой линии, часы. |
| `clarification_h` | Суммарное время уточнений | `float64` | Время уточнений в часах. |
| `resolution` | Результат работ | `string` | Результат обработки обращения. |
| `resolved_line` | Кем решен (группа) | `string` | Группа или линия, закрывшая обращение. |
| `clarifications_cnt` | Количество уточнений | `Int64` | Количество циклов уточнения. |
| `total_work_h` | Сумма line_1..4_work_h | `float64` | Общее время работы всех линий. |
| `total_react_h` | Сумма line_1..4_react_h | `float64` | Общее время реакции всех линий. |
| `lines_count` | Число линий с react_h > 0 ИЛИ work_h > 0 | `int64` | Количество фактически задействованных линий. |
| `is_multiline` | lines_count > 1 | `int64` | 1, если обращение прошло через несколько линий. |
| `lines_count_changed` | Новый lines_count != прежнего расчёта только по work_h | `int64` | 1, если исправление правила изменило число линий. |
| `is_overdue` | Просрочен?* == «Просрочен» | `int64` | 1 для просроченного обращения. |
| `is_high_clarifications` | clarifications_cnt >= 2 | `int64` | 1 при двух и более уточнениях. |
| `total_lifecycle_h` | (fact_dt - reg_dt) в часах | `float64` | Неотрицательная длительность жизненного цикла; малые отрицательные расхождения обнулены, крупные оставлены пропуском. |
| `has_datetime_anomaly` | fact_dt < reg_dt | `int64` | 1 при временной аномалии фактического завершения. |

## Правила качества и ограничения

- Времена линий, отсутствующие в источнике, интерпретируются как отсутствие активности и заменяются нулём.
- `fact_sla_h` сохраняет исходный ноль как валидное исходное значение. Только настоящий пропуск может быть восстановлен суммой времен линий.
- Если `fact_dt < reg_dt` не более чем на 5 минут, `total_lifecycle_h` принимается равным 0. При большем расхождении длительность остаётся пропуском. В обоих случаях `has_datetime_anomaly = 1`.
- `category_grouped` рассчитана по частотам всего входного набора и подходит для описательного анализа этого среза; при изменении входных данных состав top-15 может измениться.
- Средние и медианы SLA в аналитике рассчитаны только по непустым `fact_sla_h`.


## Пропуски, приближения и ограничения

- Пропущенный фактический SLA не становится нулём. Он восстанавливается суммой времени реакции и работы линий только при ненулевой сумме; иначе остаётся пропуском.
- Исходный нулевой `fact_sla_h` сохраняется и учитывается отдельно для закрытых обращений.
- Для временной аномалии до 5 минут длительность жизненного цикла принимается равной нулю. При большем отрицательном расхождении длительность не рассчитывается.
- Отсутствующие времена линий считаются отсутствием активности линии и используются как 0 при подсчёте линий.
- Риск-скор — описательный составной показатель, а не статистическая вероятность нарушения SLA.
- Укрупнение top-15 зависит от частот текущего входного набора.

## Итоговая проверка качества

In [5]:
closed_mask = clean_df["status"].str.casefold().eq("закрыт")
missing_by_column = clean_df.isna().sum()
missing_by_column = missing_by_column[missing_by_column.gt(0)].sort_values(ascending=False)

quality_check = pd.Series({
    "rows": len(clean_df),
    "unique_ticket_ids": clean_df["ticket_id"].nunique(dropna=True),
    "duplicate_ticket_ids": clean_df["ticket_id"].duplicated(keep=False).sum(),
    "duplicate_rows": clean_df.duplicated().sum(),
    "columns_with_missing": len(missing_by_column),
    "total_missing_cells": int(clean_df.isna().sum().sum()),
    "datetime_anomalies": int(clean_df["has_datetime_anomaly"].sum()),
    "closed_zero_fact_sla": int((closed_mask & clean_df["fact_sla_h"].eq(0)).sum()),
    "missing_fact_sla": int(clean_df["fact_sla_h"].isna().sum()),
    "multiline_tickets": int(clean_df["is_multiline"].sum()),
    "changed_lines_count": int(clean_df["lines_count_changed"].sum()),
}, name="value")

display(quality_check.to_frame())
print("\nПропуски по полям:")
display(missing_by_column.rename("missing_count").to_frame())

assert quality_check["rows"] == quality_check["unique_ticket_ids"], "ID должны быть уникальны"
assert quality_check["duplicate_rows"] == 0, "Полные дубли не ожидаются"
assert clean_df["category_original"].nunique() == 43
print(f"\nПроверки пройдены. Итоговый файл: {OUTPUT_CSV_PATH.resolve()}")

                      value
rows                   1931
unique_ticket_ids      1931
duplicate_ticket_ids      0
duplicate_rows            0
columns_with_missing      0
total_missing_cells       0
datetime_anomalies        0
closed_zero_fact_sla    354
missing_fact_sla          0
multiline_tickets       645
changed_lines_count      98

Пропуски по полям:
Empty DataFrame
Columns: [missing_count]
Index: []

Проверки пройдены. Итоговый файл: /Users/ilya/Documents/ChatGPT/Postcode хакатон/final.csv
